In [ ]:
from pathlib import Path

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from core import Config
import pandas as pd

config = Config()
DATA_PATH: Path = config.training_dir / "imputed_log_transformed_data.csv"

all_data: pd.DataFrame = pd.read_csv(DATA_PATH, index_col=0)
cat_cols: list[str] = ["Instrument", "Date", "TR.GICSSectorCode", "TR.HQCountryCode"]
all_data[cat_cols] = all_data[cat_cols].astype("category")

In [ ]:
training_data = all_data.copy()
training_data.drop(
    training_data[training_data["TR.UpstreamScope3PurchasedGoodsAndServices"].isna()].index,
    inplace=True)
y: pd.DataFrame = training_data['TR.UpstreamScope3PurchasedGoodsAndServices'].to_frame()
X: pd.DataFrame = training_data.drop('TR.UpstreamScope3PurchasedGoodsAndServices', axis=1)

In [66]:
from typing import cast
from sklearn.model_selection import GroupKFold
from catboost import Pool, CatBoostRegressor

gkf: GroupKFold = GroupKFold(n_splits=5)
fold_results: list[dict] = []

for fold_index, (train_index, val_index) in enumerate(gkf.split(X, y, groups=X['Instrument']), start=1):
    print(f"Fold {fold_index}")
    X_train: pd.DataFrame = cast(pd.DataFrame, X.iloc[train_index])
    X_val: pd.DataFrame = cast(pd.DataFrame, X.iloc[val_index])
    y_train: pd.Series[float] = cast(pd.Series, y.iloc[train_index])
    y_val: pd.Series[float] = cast(pd.Series, y.iloc[val_index])

    train_pool: Pool = Pool(
        data=X_train,
        label=y_train,
        cat_features=cat_cols
    )

    val_pool: Pool = Pool(
        data=X_val,
        label=y_val,
        cat_features=cat_cols
    )

    model = CatBoostRegressor(
        iterations=50,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        use_best_model=True,
        verbose=False
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
    )

    y_val_pred: list[float] = model.predict(val_pool)

    rmse: np.ndarray = np.sqrt(mean_squared_error(y_val, y_val_pred))
    mae: float = cast(float, mean_absolute_error(y_val, y_val_pred))
    r2: float = r2_score(y_val, y_val_pred)

    fold_results.append(
        {
            "model_name": "catboost_baseline",
            "fold": fold_index,
            "sector": "All",
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
        }
    )

    # sector‑level metrics
    val_sectors: pd.Series = X_val["TR.GICSSectorCode"].astype("category")

    val_sectors = X_val["TR.GICSSectorCode"]
    val_df: pd.DataFrame = pd.DataFrame(
        {
            "y_true": y_val.to_numpy().ravel(),
            "y_pred": y_val_pred,
            "sector": val_sectors.to_numpy().ravel(),
        }
    )

    for sector, grp in val_df.groupby("sector"):
        rmse = np.sqrt(mean_squared_error(grp["y_true"], grp["y_pred"]))
        mae = cast(float, mean_absolute_error(grp["y_true"], grp["y_pred"]))
        r2 = r2_score(grp["y_true"], grp["y_pred"])

        fold_results.append(
            {
                "model_name": "catboost_baseline",
                "fold": fold_index,
                "sector": sector,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
            }
        )

metrics_df: pd.DataFrame = pd.DataFrame(fold_results)
metrics_df

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


,model_name,fold,sector,rmse,mae,r2
0,catboost_baseline,1,All,57287356.80419,53880925.49475,-3.53874
1,catboost_baseline,1,10,65218119.81760,61680678.35480,-0.16721
2,catboost_baseline,1,15,52534094.62949,52410125.25608,-176.01020
3,catboost_baseline,1,20,53643057.47654,53532132.66857,-203.57363
4,catboost_baseline,1,25,49964076.45367,48306733.98431,-3.72752
5,catboost_baseline,1,30,60609628.82837,54296379.41283,-0.71051
6,catboost_baseline,1,35,53910500.72801,53853866.45604,-342.59457
7,catboost_baseline,1,40,68813951.99089,56567448.28749,-1.37479
8,catboost_baseline,1,45,53373558.49902,53257451.40622,-185.14698
9,catboost_baseline,1,50,54529885.75211,54521897.44418,-1650.97384


In [67]:
METRICS_PATH: Path = config.results_dir / "metrics_catboost_baseline_sector_level.csv"
metrics_df.to_csv(METRICS_PATH, index=False)